# Imports

In [284]:
import numpy as np
import itertools
import functools
from numpy import random as random

In [168]:
rng = random.default_rng()

### Let's start with constant h and see what happens

In [ ]:
def corrs(sigma):
    ''' 
    Input:
        - sigma: a (B,N) matrix of the response of N neurons at B time indices, where sigmat[b,n] corresponds to the response of the nth neuron at time b

    Output:
        - corrs: a (B,N*(N-1)/2) matrix of the correlation of the N neurons at B indices, for all corrs[b,i*j/2] where i < j
    '''

    

def pt(sigmat,corr,use_X,**X):
    '''
    Input:
        - sigmat: a (N,) vector of the response of N neurons at a particular time t, where sigmat[i] corresponds to the response of the ith neuron
        - corr: a (N*(N-1)/2) vector containing correlations of those N neurons at time t, where corr[i*j/2] corresponds to the product of the response of the ith and jth neurons for i < j
        - X: a (N + N*(N-1)/2 , ) vector containing:
            - h: (B*N,) vector of the time-dependent field, where h[N*t + n] corresponds to the time dependent field of the nth neuron at time t
            - J: (N(N-1)/2,) vector of the fixed couplings between two neurons

    Output:
        pt: the probability of that state at that time, given the parameters
    '''
    
    N = sigmat.shape[0]
    NN = corr.shape[0]

    if use_X == True:
        X_ = X['X']
    else:
        X_ = np.concatenate((X['h'],X['J']))
    

    pt = np.exp(-1*(sigmat@X_))
    
    return pt

def observables(sigmab):
    '''
    Input:
        - sigmab: an (N,) vector of binarized neural responses at time b; sigmab[n] is the response of neuron n
    
    Output:
        - observables: (N + N*(N-1)/2 , ) vector of the observables at that point in time
    '''
    corr = np.outer(sigmab,sigmab)
    corr_vec = corr[np.tril_indices(sigmab.shape[0],-1)]
    return np.concatenate((sigmab,corr_vec))

def P_bar(sigma):
    ''' 
    Input:
        sigma: a (B,N) matrix of binarized neural responses, where sigma[b,n] is the response of neuron n at time b

    Output:
        P_b: a (N + N*(N-1)/2 , ) vector of the average of the observables given the neural data
    '''
    B,N = sigma.shape
    P_b = np.fromiter(map(observables,sigma),dtype = np.dtype((np.float32,int(N + N*(N-1)/2))),count=B).mean(axis=0)

    return P_b

def QX(X,N):
    '''
    Input:
        - X: a (N + N*(N-1)/2 , ) vector of the parameters of the model
    
    Output:
        Q: a (N + N*(N-1)/2 , ) vector of the model averages of the observables.
    '''

    D = X.shape[0]

    combs = np.fromiter(itertools.product(range(2),repeat = N),dtype=np.dtype((np.float32,N)),count=2**N)

    weighted_observables = map(lambda x : observables(x)*pt(x,use_X = True,X=X), combs)

    wo_np = np.fromiter(weighted_observables,dtype=np.dtype((np.float32,D)),count=2**N)

    Q = wo_np.sum(axis=0)

    return Q

def QMC(sigma,M):
    '''
    Input:
        - sigma: a (B,N) matrix of binarized neural responses, where sigma[b,n] is the response of neuron n at time b
        - M: the number of times to sample from sigma
    
    Output:
        QMC: a montecarlo approximation of Q
    '''
    MC = rng.choice(sigma,M,replace=True)
    QMC = map(observables,MC)
    QMC = np.fromiter(QMC,dtype = np.dtype((np.float32,int(N + N*(N-1)/2))),count=M)
    QMC = QMC.mean(axis=0)

    return QMC

def susc_bar(sigma):
    '''
    Input:
        - sigma: a (B,N) matrix of binarized neural responses, where sigma[b,n] is the response of neuron n at time b
    
    Output:
        - X_bar: a (D,D) matrix consisting of the mean of the products of observables, subtracted from the product of the mean of each corresponding observable
            where D = N + N*(N-1)/2
    '''
    B,N = sigma.shape
    all_obs = np.fromiter(map(observables,sigma),dtype = np.dtype((np.float32,int(N + N*(N-1)/2))),count=B)
    obs_prod_bar = all_obs.T@all_obs / B
    P_ = P_bar(sigma)
    prod_obs_bar = np.outer(P_,P_)
    return obs_prod_bar - prod_obs_bar

def epsilon(P_bar,X_bar,Q,B):
    D = P_bar.shape[0]
    diff = P_bar - Q
    inv_sus = np.linalg.inv(X_bar)
    return np.sqrt(np.abs((2*B / D)*(diff@inv_sus@diff)))

def Z(X)

In [ ]:
class fixed_h_MaxEnt:
    def __init__(
        self,
        X0_init = 'random',
        a0 = 1,
        del_p = np.float32(1.05),
        del_n = np.sqrt(2,dtype=np.float32)
    ):
        self.a0 = a0
        self.X0_init = 'random'
        self.del_p = del_p
        self.del_n = del_n
        self.X = None

    def fit(self,sigma):
        B,N = sigma.shape
        D = int(N+N*(N-1)/2)
        if self.X0_init == 'random':
            X0 = random.rand(D)
        else:
            print('not implemented yet!')
            return None

        Q0 = QX(X0,N)
        P_ = P_bar(sigma)
        X_ = susc_bar(sigma)
        e0 = epsilon(P_,X_,Q0,B)
        inv_X_ = np.linalg.inv(X_)

        et = e0
        Qt = Q0
        Xt = X0
        at = self.a0
        while et >= 1:
            Mt = np.min((B/et**2,B)).astype(int)
            Xt_1 = at*Xt@(P_ - Qt)
            Qt_1 = QMC(sigma,Mt)
            et_1 = epsilon(P_,X_,Qt_1,B)
            if et_1 < et:
                Qt = Qt_1
                Xt = Xt_1
                et = et_1
                at_1 = at*self.del_p
                at = at_1
            else:
                at_1 = at*self.del_n
        
        self.X_final = Xt

        return Xt
    
    def predict(self,sigmat):
        if self.X == None:
            print('Hasn\'t been fit yet!')
            return None
        return pt(sigmat,use_X = True,X=self.X)
            

In [383]:
B = 50
N = 20
D = int(N + N*(N-1)/2)
t = 2
sigma = rng.choice(range(2),(B,N),replace=True)
X = rng.random(D)
P_ = P_bar(sigma)
X_ = susc_bar(sigma)

In [396]:
combs = np.fromiter(itertools.product(range(2),repeat = N),dtype=np.dtype((np.float32,N)),count=2**N)

weighted_observables = np.fromiter(map(lambda x : pt(x,use_X = True,X=X), combs),dtype=np.float32,count=2**N)

#ZX = functools.reduce(lambda x,y : x + y, weighted_observables)

KeyboardInterrupt: 